# 04 · Validar a qualidade da silver

Antes de publicar a gold, conferimos se a silver está confiável. O notebook **não decide nada sozinho**: ele mede, mostra o resultado e **publica um task value** (`qualidade_ok`). Quem decide o caminho é o job, com uma tarefa **If/else** (aula 16).

Regras verificadas:
1. Toda tabela da silver tem pelo menos 1 linha.
2. Nenhuma chave primária está duplicada.
3. O percentual de empréstimos sem valor (`loan_amount` nulo depois da limpeza) não passa do limite `limite_nulos_pct`.

> Dica para a aula: para testar o caminho "falso" do If/else, rode o job com `limite_nulos_pct = 0`.

In [ ]:
dbutils.widgets.text("catalogo", "growdev", "Catálogo")
dbutils.widgets.text("limite_nulos_pct", "5", "Limite de nulos (%)")

CATALOGO = dbutils.widgets.get("catalogo")
LIMITE_NULOS_PCT = float(dbutils.widgets.get("limite_nulos_pct"))

In [ ]:
from pyspark.sql import functions as F

chaves = {
    "account_balances": "balance_id",
    "accounts": "account_id",
    "customers": "customer_id",
    "loans": "loan_id",
}

resultados = []  # lista de dicionários: uma linha por regra verificada

for tabela, chave in chaves.items():
    df = spark.table(f"{CATALOGO}.silver.{tabela}")
    linhas = df.count()
    duplicadas = df.groupBy(chave).count().filter(F.col("count") > 1).count()
    resultados.append({"regra": f"{tabela}: tem linhas", "valor": float(linhas), "ok": linhas > 0})
    resultados.append({"regra": f"{tabela}: chaves duplicadas", "valor": float(duplicadas), "ok": duplicadas == 0})

df_loans = spark.table(f"{CATALOGO}.silver.loans")
pct_nulos = round(100 * df_loans.filter(F.col("loan_amount").isNull()).count() / df_loans.count(), 2)
resultados.append({
    "regra": f"loans: % de loan_amount nulo <= {LIMITE_NULOS_PCT}",
    "valor": pct_nulos,
    "ok": pct_nulos <= LIMITE_NULOS_PCT,
})

display(spark.createDataFrame(resultados))

In [ ]:
qualidade_ok = all(r["ok"] for r in resultados)

# Task values: "recados" que esta tarefa deixa para as próximas tarefas do job
dbutils.jobs.taskValues.set(key="qualidade_ok", value=qualidade_ok)
dbutils.jobs.taskValues.set(key="pct_nulos_loan_amount", value=pct_nulos)

print(f"Qualidade OK? {qualidade_ok} | % de loan_amount nulo: {pct_nulos}%")